In [2]:
print("hello")

hello


In [3]:
from dotenv import load_dotenv
import os
load_dotenv("../.env")

for k in ["GROQ_API_KEY", "SERPAPI_API_KEY", "PINECONE_API_KEY","PINECONE_INDEX_NAME","PINECONE_NAMESPACE"]:
    print(k, "->", bool(os.getenv(k)))


GROQ_API_KEY -> True
SERPAPI_API_KEY -> True
PINECONE_API_KEY -> True
PINECONE_INDEX_NAME -> True
PINECONE_NAMESPACE -> True


In [4]:
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

client = Groq()

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain an AI agent in 2 sentences."
        }
    ],
    max_tokens=200,
)

print(response.choices[0].message.content)
print(response.usage)

An AI agent is a software entity that perceives its environment through data inputs, processes that information using learned or programmed models, and takes actions to achieve specific goals. It continuously iterates this perception‑decision‑action loop, adapting its behavior over time based on feedback or new observations.
CompletionUsage(completion_tokens=104, prompt_tokens=80, total_tokens=184, completion_time=0.218118561, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=38), prompt_time=0.003157617, prompt_tokens_details=None, queue_time=0.316407524, total_time=0.221276178)


In [5]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path("../.env").resolve()
print("Found .env:", env_path.is_file(), "| Path:", env_path)

load_dotenv(env_path, override=True)
api_key = os.getenv("ANTHROPIC_API_KEY", "").strip()
print("Key loaded:", bool(api_key), "| Length:", api_key)

Found .env: True | Path: D:\Langchain\Enterprise-AI-Agents\.env
Key loaded: False | Length: 


In [ ]:
import os, requests
from dotenv import load_dotenv
load_dotenv("../.env")

def web_search(query, num=5):
    r = requests.get(
        "https://serpapi.com/search",
        params={
            "engine": "google",
            "q": query,
            "num": num,
            "api_key": os.getenv("SERPAPI_API_KEY"),
        },
        timeout=30,
    )
    print("Status:", r.status_code)   # keep this while debugging
    r.raise_for_status()
    return r.json().get("organic_results", [])

for item in web_search("what is agentic AI"):
    print(item["title"], "|", item["link"])
    print(item.get("snippet", ""), "\n")


Status: 200
What is Agentic AI? | https://www.ibm.com/think/topics/agentic-ai
Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of ai agents—machine learning ... 

Agentic AI, explained | https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained
AI agents or agentic AI, a new breed of AI systems that are semi- or fully autonomous and thus able to perceive, reason, and act on their own. 

What is agentic AI? Definition and differentiators | https://cloud.google.com/discover/what-is-agentic-ai
Agentic AI is an advanced form of artificial intelligence focused on autonomous decision-making and action. Unlike traditional AI, which ... 

What is Agentic AI? - AWS - Amazon.com | https://www.google.com/goto?url=CAESZwHEIkXXnysNbms5CfaNGzGrb2oIjIw5XMYFt8BJRG2XjpWLh7Uzpp_5_XxWQu_9AL1HknHH2xXWur6XJLOIYeIwBQqG8YlGRuYdpdABsH1nS_X08QTkc1Z1gbXtJiVtL_hN7rLYfQ8
Agentic AI is an autonomous AI system that can act independent

In [7]:
from dotenv import load_dotenv
from pinecone import Pinecone
load_dotenv("../.env")

pc = Pinecone(api_key="") #IF API KEY is present in .env no needt to put here
print(pc.list_indexes().names())


['enterprise-ai-agents']


In [8]:
import os
os.environ.setdefault("USER_AGENT", "Mozilla/5.0 Agentic-RAG-Demo")

from langchain_community.document_loaders import WebBaseLoader

def load_web_pages(urls, user_agent="Mozilla/5.0 Agentic-RAG-Demo"):
    """
    Load one or more web pages and return a list of LangChain Documents.

    Args:
        urls: a single URL (str) or a list of URLs
        user_agent: User-Agent header sent with the request

    Returns:
        list of Document objects (one per URL)
    """
    # Allow passing a single string
    if isinstance(urls, str):
        urls = [urls]

    loader = WebBaseLoader(
        web_paths=urls,
        requests_kwargs={"headers": {"User-Agent": user_agent}},
    )

    try:
        docs = loader.load()
    except Exception as e:
        print(f"Error loading pages: {e}")
        return []

    return docs

docs = load_web_pages("https://docs.langchain.com/oss/python/langgraph/agentic-rag")
print(docs)

C:\Users\mahat\AppData\Local\Temp\ipykernel_20836\2577242680.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


Error loading pages: No module named 'bs4'
[]


In [9]:
import re
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader

def load_pdf(
    file_path,
    extract_images=False,
    clean=True,
    min_chars=20,
    verbose=True,
):
    """
    Load a PDF using PyMuPDFLoader and return a list of LangChain Documents
    (one per page).

    Args:
        file_path (str | Path): path to the PDF file.
        extract_images (bool): run OCR on images inside the PDF
                               (needs: pip install rapidocr-onnxruntime).
        clean (bool): normalize whitespace in the extracted text.
        min_chars (int): pages with fewer characters than this are dropped.
        verbose (bool): print a summary of what was loaded.

    Returns:
        list[Document]: Documents with cleaned text and enriched metadata.
    """
    # 1. Validate the file
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {path}")
    if path.suffix.lower() != ".pdf":
        raise ValueError(f"Not a PDF file: {path.name}")

    # 2. Load with PyMuPDFLoader
    try:
        loader = PyMuPDFLoader(str(path), extract_images=extract_images)
        docs = loader.load()
    except Exception as e:
        raise RuntimeError(f"Failed to load '{path.name}': {e}")

    total_pages = len(docs)
    cleaned_docs = []
    skipped_pages = []

    # 3. Clean, filter and enrich each page
    for doc in docs:
        text = doc.page_content

        if clean:
            text = re.sub(r"[ \t]+", " ", text)       # collapse spaces and tabs
            text = re.sub(r"\n{3,}", "\n\n", text)    # max one blank line
            text = text.strip()

        page_index = doc.metadata.get("page")

        # Skip blank or image-only pages
        if len(text) < min_chars:
            skipped_pages.append(page_index)
            continue

        doc.page_content = text
        doc.metadata["file_name"] = path.name
        if isinstance(page_index, int):
            doc.metadata["page_number"] = page_index + 1   # starts at 1

        cleaned_docs.append(doc)

    # 4. Summary and warnings
    if verbose:
        print(f"File          : {path.name}")
        print(f"Total pages   : {total_pages}")
        print(f"Pages kept    : {len(cleaned_docs)}")
        print(f"Pages skipped : {len(skipped_pages)}")

    if not cleaned_docs:
        print(
            "WARNING: no text was extracted. The PDF may be scanned (image-only). "
            "Try extract_images=True or an OCR-based approach."
        )

    return cleaned_docs


In [10]:

docs = load_pdf("C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf")

print(len(docs))                    # number of pages with text
print(docs[0].page_content[:300])   # first 300 characters of page 1
print(docs[0].metadata)


File          : Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf
Total pages   : 19
Pages kept    : 19
Pages skipped : 0
19
IEEE TRANSACTIONS ON ARTIFICIAL INTELLIGENCE, VOL. 6, NO. 12, DECEMBER 2025
3165
Generative Agents in Agent-Based Modeling:
Overview, Validation, and Emerging Challenges
Carlo Adornetto
, Adrian Mora
, Kai Hu
, Leticia Izquierdo Garcia
, Parfait Atchade-Adelomou
,
Gianluigi Greco
, Member, IEEE, Lui
{'producer': 'Adobe LiveCycle PDF Generator; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-23T23:20:44+05:30', 'source': 'C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf', 'file_path': 'C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf', 'total_pages': 19, 'format': 'PDF 1.7', 'title': 'Generative Agents

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(
    docs,
    chunk_size=500,
    chunk_overlap=100,
    add_chunk_id=True,
    verbose=True,
):
    """
    Split LangChain Documents into smaller chunks.

    Args:
        docs (list[Document]): Documents from a loader (PDF, web, etc.).
        chunk_size (int): maximum characters per chunk.
        chunk_overlap (int): characters repeated between neighbouring chunks.
        add_chunk_id (bool): add a chunk number to each chunk's metadata.
        verbose (bool): print a summary.

    Returns:
        list[Document]: smaller Documents with the original metadata kept.
    """
    # 1. Validate input
    if not docs:
        print("WARNING: no documents to split.")
        return []
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size.")

    # 2. Create the splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],  # try paragraphs first, then lines, sentences, words
    )

    # 3. Split (metadata is copied to every chunk)
    chunks = splitter.split_documents(docs)

    # 4. Add a chunk number to the metadata
    if add_chunk_id:
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = i

    # 5. Summary
    if verbose:
        lengths = [len(c.page_content) for c in chunks]
        print(f"Input documents : {len(docs)}")
        print(f"Chunks created  : {len(chunks)}")
        print(f"Avg chunk size  : {sum(lengths) // len(lengths)} characters")
        print(f"Min / Max size  : {min(lengths)} / {max(lengths)}")

    return chunks

In [12]:
chunks = split_documents(docs)

print(len(chunks))

Input documents : 19
Chunks created  : 289
Avg chunk size  : 458 characters
Min / Max size  : 170 / 499
289


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def get_embedding_model(model_name="sentence-transformers/all-MiniLM-L6-v2"):
    """Create an embedding model that runs on your own computer."""
    return HuggingFaceEmbeddings(
        model_name=model_name,
        encode_kwargs={"normalize_embeddings": True},  # helps similarity search
    )


def embed_chunks(chunks, embedding_model=None, verbose=True):
    """
    Convert chunks (Documents) into embedding vectors.

    Args:
        chunks (list[Document]): chunks from split_documents().
        embedding_model: a LangChain embeddings object (created if None).
        verbose (bool): print a summary.

    Returns:
        tuple: (vectors, embedding_model)
            vectors = list of lists of floats, one per chunk
    """
    if not chunks:
        print("WARNING: no chunks to embed.")
        return [], embedding_model

    if embedding_model is None:
        embedding_model = get_embedding_model()

    # 1. Extract the text of each chunk
    texts = [chunk.page_content for chunk in chunks]

    # 2. Convert all texts to vectors
    vectors = embedding_model.embed_documents(texts)

    # 3. Summary
    if verbose:
        print(f"Chunks embedded   : {len(vectors)}")
        print(f"Vector dimensions : {len(vectors[0])}")

    return vectors, embedding_model

# Embedding vector
vectors, embedding_model = embed_chunks(chunks)
print(len(vectors))        # same as number of chunks
print(len(vectors[0]))     # 384 for all-MiniLM-L6-v2
print(vectors[0][:5])      # first 5 numbers of the first vector

d:\Langchain\Enterprise-AI-Agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4963.56it/s]


Chunks embedded   : 289
Vector dimensions : 384
289
384
[-0.051384538412094116, -0.0479910671710968, 0.013866160064935684, 0.03520385921001434, -0.024448100477457047]


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()   # reads the .env file in your project folder

PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "enterprise-ai-agents")
PINECONE_NAMESPACE = os.getenv("PINECONE_NAMESPACE", "RagQuestions")

if not os.getenv("PINECONE_API_KEY"):
    raise EnvironmentError("PINECONE_API_KEY is missing in your .env file.")

In [15]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore


def clean_metadata(chunks):
    """Pinecone metadata allows only strings, numbers, booleans, lists of strings."""
    for chunk in chunks:
        cleaned = {}
        for key, value in chunk.metadata.items():
            if value is None:
                continue
            if isinstance(value, (str, int, float, bool)):
                cleaned[key] = value
            else:
                cleaned[key] = str(value)
        chunk.metadata = cleaned
    return chunks


def store_in_pinecone(
    chunks,
    embedding_model,
    index_name="enterprise-ai-agents",
    namespace="RagQuestions",
    cloud="aws",
    region="us-east-1",
    metric="cosine",
    batch_size=100,
    verbose=True,
):
    """Embed chunks and store them in a Pinecone index."""
    if not chunks:
        raise ValueError("No chunks to store.")

    api_key = os.environ.get("PINECONE_API_KEY")
    if not api_key:
        raise EnvironmentError("Set the PINECONE_API_KEY in your .env file.")

    # 1. Get embedding dimension from the model (384 for MiniLM)
    dimension = len(embedding_model.embed_query("dimension check"))

    # 2. Create the index if it does not exist
    pc = Pinecone(api_key=api_key)
    existing = [idx["name"] for idx in pc.list_indexes()]

    if index_name not in existing:
        if verbose:
            print(f"Creating index '{index_name}' (dimension={dimension})...")
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(cloud=cloud, region=region),
        )
        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)
    else:
        info = pc.describe_index(index_name)
        if info.dimension != dimension:
            raise ValueError(
                f"Index '{index_name}' has dimension {info.dimension}, but your "
                f"model produces {dimension}. Use a new index name."
            )
        if verbose:
            print(f"Using existing index '{index_name}'.")

    # 3. Clean metadata
    chunks = clean_metadata(chunks)

    # 4. Embed and upload
    vectorstore = PineconeVectorStore(
        index=pc.Index(index_name),
        embedding=embedding_model,
        namespace=namespace,
    )

    ids = [f"{chunk.metadata.get('file_name', 'doc')}-{i}" for i, chunk in enumerate(chunks)]
    vectorstore.add_documents(chunks, ids=ids, namespace=namespace, batch_size=batch_size)

    if verbose:
        print(f"Stored {len(chunks)} chunks in '{index_name}' (namespace '{namespace}').")

    return vectorstore

In [19]:
vectorstore = store_in_pinecone(
    chunks,
    embedding_model,
    index_name=PINECONE_INDEX_NAME,
    namespace=PINECONE_NAMESPACE,
)

results = vectorstore.similarity_search("What is the main topic?", k=3)

Creating index 'enterprise-ai-agents' (dimension=384)...
Stored 289 chunks in 'enterprise-ai-agents' (namespace 'RagQuestions').


In [21]:
results = vectorstore.similarity_search("What is the main topic?", k=3)
print(results)


[Document(id='Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf-33', metadata={'author': '', 'chunk_id': 33.0, 'creationDate': "D:20251123232044+05'30'", 'creationdate': '2025-11-23T23:20:44+05:30', 'creator': 'LaTeX with hyperref', 'file_name': 'Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf', 'file_path': 'C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': "D:20251126142426-05'00'", 'moddate': '2025-11-26T14:24:26-05:00', 'page': 2.0, 'page_number': 3.0, 'producer': 'Adobe LiveCycle PDF Generator; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'source': 'C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf', 'subject': 'IEEE Transactions on Artificial Intelligence;2025;6;12;10.1109/TAI.2025.35

In [23]:
def get_retriever(embedding_model, k=4):
    vectorstore = PineconeVectorStore(
        index_name=os.getenv("PINECONE_INDEX_NAME"),
        embedding=embedding_model,
        namespace=os.getenv("PINECONE_NAMESPACE"),
    )
    return vectorstore.as_retriever(search_kwargs={"k": k})

retriever = get_retriever(embedding_model, k=4)

In [24]:
from langchain_core.prompts import ChatPromptTemplate
# 2. Format retrieved chunks as context
def format_context(docs):
    parts = []
    for i, d in enumerate(docs, start=1):
        file = d.metadata.get("file_name", "unknown")
        page = d.metadata.get("page_number", "?")
        parts.append(f"[{i}] (source: {file}, page {page})\n{d.page_content}")
    return "\n\n".join(parts)

PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the context provided below.\n"
     "If the answer is not in the context, say \"I could not find this in the "
     "documents.\" Do not guess.\n"
     "Mention the source numbers like [1] or [2] that support your answer.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

In [31]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
    temperature=0,
    max_tokens=1000,
)

In [32]:
from langchain_core.output_parsers import StrOutputParser
def ask(question, retriever):
    docs = retriever.invoke(question)

    if not docs:
        return {"answer": "No relevant documents found.", "sources": []}

    chain = PROMPT | llm | StrOutputParser()
    answer = chain.invoke({
        "context": format_context(docs),
        "question": question,
    })

    sources = [
        {
            "file": d.metadata.get("file_name"),
            "page": d.metadata.get("page_number"),
            "preview": d.page_content[:150] + "...",
        }
        for d in docs
    ]
    return {"answer": answer, "sources": sources}

In [33]:
result = ask("What is evolving egents?", retriever)
print(result["answer"])
print("\nSources:")

Evolving agents are agents whose behaviors are continuously refined and adapted through evolutionary processes—such as natural‑selection‑style selection, mutation, and recombination—typically implemented with bio‑inspired evolutionary algorithms (EAs) like genetic algorithms. In agent‑based modeling (ABM) this approach is used to create agents that can adapt, evolve, and improve their strategies over time, often in conjunction with machine‑learning techniques【2】.

Sources:


In [34]:
for s in result["sources"]:
    print(f"- {s['file']}, page {s['page']}: {s['preview']}")

- Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf, page 3.0: natural selection by reﬁning agent behaviors through selection,
mutation, and recombination [18], [19]. EAs encode solutions
as individuals, evolving ...
- Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf, page 3.0: (of social science) ABMs and the models designed with rule-
based, evolutionary, or reinforcement learning approaches. For
ABMs speciﬁcally designed w...
- Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf, page 15.0: structures, but they often rely on static or probability-based
adoption rules [80]. GAs, in contrast, simulate the emer-
gence, adoption, and dissemin...
- Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf, page 18.0: 3182
IEEE TRANSACTIONS ON ARTIFICIAL INTELLIGENCE, VOL. 6, NO. 12, DECEMBER 2025
[18] J. H. Holland, Adaptation in Natural a

In [ ]:
# We have succefully build small rag system using pinecone amd Groq 